In [ ]:
# Base de datos en DATA (GITHUB)

import pandas as pd

# 1. Cargar la base de datos antigua
df_antiguo = pd.read_csv('../../data/dataset_features_temperatura.csv')

# 2. Las 3 variables estrella que queremos analizar
variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']

# Asegurar que la columna de temperatura y las variables existen
columnas_interes = ['temperature'] + [v for v in variables_clave if v in df_antiguo.columns]

print("📊 --- ANALIZANDO LA BASE DE DATOS DE ENTRENAMIENTO --- 📊")
print("Agrupando por temperatura y calculando la media de cada variable:\n")

# 3. Agrupar por temperatura y sacar la media de los datos raw antiguos
analisis = df_antiguo[columnas_interes].groupby('temperature').mean()

print(analisis.to_string())

📊 --- ANALIZANDO LA BASE DE DATOS DE ENTRENAMIENTO --- 📊
Agrupando por temperatura y calculando la media de cada variable:

             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                      112.191273    0.022558       107.050472
18.0                      171.445088    0.034561       108.607002
24.0                       81.440384    0.016410       109.557392
30.0                      400.307246    0.080852        49.561525
34.0                       81.975337    0.016528       107.893959
36.0                      251.160997    0.050711        81.053986
38.0                      242.932699    0.049037        61.208049
40.0                      436.429851    0.088410        59.840642
42.0                      206.243056    0.041622        53.255018
44.0                      580.680019    0.116941        44.264487
46.0                     1862.537242    0.375492        30.060902
48.0              

In [4]:
# Base de datos Whastapp Hugo

import pandas as pd

# 1. Cargar la base de datos antigua
df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

# 2. Las 3 variables estrella que queremos analizar
variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']

# Asegurar que la columna de temperatura y las variables existen
columnas_interes = ['temperature'] + [v for v in variables_clave if v in df_antiguo.columns]

print("📊 --- ANALIZANDO LA BASE DE DATOS DE ENTRENAMIENTO --- 📊")
print("Agrupando por temperatura y calculando la media de cada variable:\n")

# 3. Agrupar por temperatura y sacar la media de los datos raw antiguos
analisis = df_antiguo[columnas_interes].groupby('temperature').mean()

print(analisis.to_string())

📊 --- ANALIZANDO LA BASE DE DATOS DE ENTRENAMIENTO --- 📊
Agrupando por temperatura y calculando la media de cada variable:

             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                     5551.464762    1.119843        29.522197
19.0                     5668.521396    1.145358        29.198023
23.0                     8539.825785    1.725401        25.177749
28.0                     4753.487311    0.961509        31.417853
33.0                     4947.778776    0.998324        30.999475
40.0                     2926.288775    0.590382        38.067776
45.0                     3091.627489    0.625079        37.079187
52.0                     5630.720661    1.137102        29.443816
58.0                     4316.924928    0.872369        33.072704
62.0                     5163.396128    1.044252        30.391921
70.0                     6642.931578    1.340847        27.653414
80.0              

In [5]:
# Vaso de carton

import sys
import pandas as pd
from pathlib import Path

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. CONFIGURAR RUTAS Y ETIQUETAS
# =============================================================================
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../../data/carton")

# Etiquetamos cada par de archivos con la temperatura del agua que usasteis
pruebas = [
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']
datos_extraidos = []

print("📡 Procesando archivos .bin del vaso de cartón para generar la tabla...")

# =============================================================================
# 3. EXTRACCIÓN Y CÁLCULO
# =============================================================================
for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # Nos quedamos con las 3 variables estrella y apuntamos su temperatura
        features_filtradas = {var: todas_features[var] for var in variables_clave}
        features_filtradas['temperature'] = p['temp']
        datos_extraidos.append(features_filtradas)
        
    except Exception as e:
        print(f"❌ Error en la muestra de {p['temp']}ºC: {e}")

# =============================================================================
# 4. IMPRESIÓN DE LA TABLA FINAL
# =============================================================================
if len(datos_extraidos) > 0:
    df_carton = pd.DataFrame(datos_extraidos)
    
    print("\n📊 --- ANALIZANDO LA BASE DE DATOS DE CARTÓN --- 📊")
    print("Agrupando por temperatura y calculando la media de cada variable:\n")
    
    # Agrupamos por temperatura y sacamos la media (como en tu captura)
    analisis_carton = df_carton.groupby('temperature')[variables_clave].mean()
    
    print(analisis_carton.to_string())
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

📡 Procesando archivos .bin del vaso de cartón para generar la tabla...

📊 --- ANALIZANDO LA BASE DE DATOS DE CARTÓN --- 📊
Agrupando por temperatura y calculando la media de cada variable:

             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                        2.450839    0.000494        90.577112
40.0                        3.322160    0.000666       117.364625
90.0                        7.386471    0.001484       180.377926


In [10]:
# Vaso de cristal

import sys
import pandas as pd
from pathlib import Path

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. CONFIGURAR RUTAS Y ETIQUETAS
# =============================================================================
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../../data/cristal")

# Etiquetamos cada par de archivos con la temperatura del agua que usasteis
pruebas = [
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']
datos_extraidos = []

print("📡 Procesando archivos .bin del vaso de cristal para generar la tabla...")

# =============================================================================
# 3. EXTRACCIÓN Y CÁLCULO
# =============================================================================
for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # Nos quedamos con las 3 variables estrella y apuntamos su temperatura
        features_filtradas = {var: todas_features[var] for var in variables_clave}
        features_filtradas['temperature'] = p['temp']
        datos_extraidos.append(features_filtradas)
        
    except Exception as e:
        print(f"❌ Error en la muestra de {p['temp']}ºC: {e}")

# =============================================================================
# 4. IMPRESIÓN DE LA TABLA FINAL
# =============================================================================
if len(datos_extraidos) > 0:
    df_carton = pd.DataFrame(datos_extraidos)
    
    print("\n📊 --- ANALIZANDO LA BASE DE DATOS DE CRISTAL --- 📊")
    print("Agrupando por temperatura y calculando la media de cada variable:\n")
    
    # Agrupamos por temperatura y sacamos la media (como en tu captura)
    analisis_carton = df_carton.groupby('temperature')[variables_clave].mean()
    
    print(analisis_carton.to_string())
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

📡 Procesando archivos .bin del vaso de cristal para generar la tabla...

📊 --- ANALIZANDO LA BASE DE DATOS DE CRISTAL --- 📊
Agrupando por temperatura y calculando la media de cada variable:

             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                        1.820800    0.000367        57.875858
40.0                        4.880010    0.000986       107.027191
90.0                        8.557909    0.001719       122.163237


In [11]:
# Vaso de plástico

import sys
import pandas as pd
from pathlib import Path

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. CONFIGURAR RUTAS Y ETIQUETAS
# =============================================================================
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../../data/plastico")

# Etiquetamos cada par de archivos con la temperatura del agua que usasteis
pruebas = [
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"temp": 15.0, "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"temp": 40.0, "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"temp": 90.0, "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']
datos_extraidos = []

print("📡 Procesando archivos .bin del vaso de plástico para generar la tabla...")

# =============================================================================
# 3. EXTRACCIÓN Y CÁLCULO
# =============================================================================
for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # Nos quedamos con las 3 variables estrella y apuntamos su temperatura
        features_filtradas = {var: todas_features[var] for var in variables_clave}
        features_filtradas['temperature'] = p['temp']
        datos_extraidos.append(features_filtradas)
        
    except Exception as e:
        print(f"❌ Error en la muestra de {p['temp']}ºC: {e}")

# =============================================================================
# 4. IMPRESIÓN DE LA TABLA FINAL
# =============================================================================
if len(datos_extraidos) > 0:
    df_carton = pd.DataFrame(datos_extraidos)
    
    print("\n📊 --- ANALIZANDO LA BASE DE DATOS DE PLÁSTICO --- 📊")
    print("Agrupando por temperatura y calculando la media de cada variable:\n")
    
    # Agrupamos por temperatura y sacamos la media (como en tu captura)
    analisis_carton = df_carton.groupby('temperature')[variables_clave].mean()
    
    print(analisis_carton.to_string())
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

📡 Procesando archivos .bin del vaso de plástico para generar la tabla...

📊 --- ANALIZANDO LA BASE DE DATOS DE PLÁSTICO --- 📊
Agrupando por temperatura y calculando la media de cada variable:

             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                        1.785570    0.000359        75.161377
40.0                        2.807597    0.000567        96.213180
90.0                        8.464673    0.001691       146.674596
